In [1]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques
Fichiers copiés dans : data


In [49]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [50]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [51]:
test_ids = df_test['Id'].copy()
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

KeyboardInterrupt: 

In [ ]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

0.12457570576748891


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

modeles = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

for key, mod in modeles.items():
    mod.fit(X_train, np.log1p(y_train))
    y_pred = mod.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
    print(f"{key:20s} : {rmse:.4f}")


LinearRegression     : 0.1246
RandomForest         : 0.1467
GradientBoosting     : 0.1375


In [ ]:
from sklearn.model_selection import cross_val_score
import sklearn.metrics
sklearn.metrics.get_scorer_names()
print(sklearn.metrics.get_scorer_names()
)

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'd2_brier_score', 'd2_log_loss_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_m

In [52]:
X,_,_=p.preprocessing(X)

In [ ]:
for key,mod in modeles.items():
    scores=cross_val_score(mod,X,np.log1p(y),cv=5,scoring='neg_mean_squared_error')
    print(f"RMSE of log-prediction, of {key:20s} is {np.sqrt(-scores.mean())}")

log-RMSE of LinearRegression     is 0.14047746751248424
log-RMSE of RandomForest         is 0.14237692001393984
log-RMSE of GradientBoosting     is 0.12527865507163707
